In [1]:
import os
os.chdir('..')

In [2]:
from src.models.architecture.wgan_gp import UpsampleConv, Generator

import torch
import torch.nn as nn

In [3]:
model = Generator()
params = torch.load("/kaggle/input/notebooks/vngtriu/wgan-gp-2000eps-n/gen_model.pt")
model.load_state_dict(params)

<All keys matched successfully>

In [4]:
parallel = nn.DataParallel(model)
model.to("cuda:0")

Generator(
  (project): Sequential(
    (0): Linear(in_features=256, out_features=16384, bias=True)
    (1): ReLU(inplace=True)
  )
  (main): Sequential(
    (0): UpsampleConv(
      (upsample): Sequential(
        (0): Upsample(scale_factor=2.0, mode='bilinear')
        (1): Conv2d(1024, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (2): BatchNorm2d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (3): LeakyReLU(negative_slope=0.2, inplace=True)
      )
      (main): Sequential(
        (0): ConvTranspose2d(1024, 512, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
        (1): BatchNorm2d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU(inplace=True)
      )
      (shortcut): Sequential(
        (0): Upsample(scale_factor=2.0, mode='nearest')
        (1): Conv2d(1024, 512, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (2): BatchNorm2d(512, eps=1e-05, momentum=0.1, affine=True, 

In [6]:
data_packs = []

for _ in range(11):
    torch.cuda.empty_cache()
    with torch.no_grad():
        noise = parallel.module.sample_latent(256, device="cuda:0")
        data = parallel(noise)
        data.to("cpu")
        data_packs.append(data)
        del data
        del noise

data = torch.cat(data_packs, dim=0)
torch.save(torch.cat(data_packs, dim=0), "final_data.pt")
print(data.shape)

torch.Size([2816, 1, 128, 128])
